# ADBA QLoRA for RTX 5090

Notebook nay fine-tune `Qwen2.5-Coder-7B-Instruct` bang QLoRA tren GPU NVIDIA RTX 5090 32GB.

Muc tieu:
- Train tren `data/train.jsonl`
- Chon checkpoint tot nhat theo `data/valid.jsonl`
- Giu `data/test.jsonl` lam holdout, khong dung trong training
- Luu adapter de copy ve may local va evaluate tiep

Khuyen nghi van hanh:
- Copy ca repo len may GPU, mo notebook nay trong Jupyter.
- Neu Windows native gap van de voi `bitsandbytes`, chuyen sang WSL2 Ubuntu tren cung may do.
- Voi 787 mau train, uu tien `3 epochs + early stopping`, khong nen ep `1200 steps` vi de overfit.

In [ ]:
# Chay cell nay neu may GPU chua co moi truong train.
# Neu server da co san torch CUDA phu hop, co the bo qua cell nay.

!pip install -U pip
!pip install -U jupyter ipywidgets datasets accelerate peft trl transformers bitsandbytes


In [ ]:
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

print('torch           :', torch.__version__)
print('cuda available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu             :', torch.cuda.get_device_name(0))
    print('compute dtype   :', 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16')


In [ ]:
# Auto-detect project root. Khuyen nghi mo notebook tu folder training/.
CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_DIR = None
for candidate in CANDIDATES:
    if (candidate / 'data' / 'train.jsonl').exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError('Khong tim thay data/train.jsonl. Hay mo notebook ben trong repo ADBA.')

TRAIN_PATH = PROJECT_DIR / 'data' / 'train.jsonl'
VALID_PATH = PROJECT_DIR / 'data' / 'valid.jsonl'
TEST_PATH = PROJECT_DIR / 'data' / 'test.jsonl'
OUT_DIR = PROJECT_DIR / 'adapters' / 'qwen25-coder-7b-adba-qlora-rtx5090'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ_LENGTH = 2048
USE_4BIT = True
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
TORCH_DTYPE = torch.bfloat16 if BF16 else torch.float16
OPTIMIZER = 'paged_adamw_8bit' if USE_4BIT else 'adamw_torch'

print('PROJECT_DIR:', PROJECT_DIR)
print('TRAIN      :', TRAIN_PATH)
print('VALID      :', VALID_PATH)
print('TEST       :', TEST_PATH)
print('OUT_DIR    :', OUT_DIR)


In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
valid_rows = load_jsonl(VALID_PATH)
test_rows = load_jsonl(TEST_PATH)

print('train rows:', len(train_rows))
print('valid rows:', len(valid_rows))
print('test rows :', len(test_rows))
print('sample keys:', list(train_rows[0].keys()))
print('sample roles:', [m['role'] for m in train_rows[0]['messages']])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=TORCH_DTYPE,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_ds = Dataset.from_list(train_rows).map(format_example, remove_columns=['messages'])
valid_ds = Dataset.from_list(valid_rows).map(format_example, remove_columns=['messages'])
test_ds = Dataset.from_list(test_rows).map(format_example, remove_columns=['messages'])

response_template = '<|im_start|>assistant\n'
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

print(train_ds)
print(valid_ds)
print('first formatted sample preview:')
print(train_ds[0]['text'][:1200])


In [ ]:
# 787 mau train la nho. 3 epochs voi early stopping hop ly hon lap 1200 steps.
training_args = SFTConfig(
    output_dir=str(OUT_DIR),
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=3,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim=OPTIMIZER,
    logging_steps=10,
    evaluation_strategy='steps',
    save_strategy='steps',
    eval_steps=25,
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=BF16,
    fp16=not BF16,
    gradient_checkpointing=True,
    packing=False,
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer


In [ ]:
train_result = trainer.train()
metrics = train_result.metrics
metrics


In [ ]:
eval_metrics = trainer.evaluate(valid_ds)
eval_metrics


In [ ]:
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

print('Saved adapter to:', OUT_DIR)
print('Files:')
for path in sorted(OUT_DIR.glob('*')):
    print(' -', path.name)


In [ ]:
# Smoke test tren vai mau holdout. Cell nay khong tinh metric, chi kiem tra chat luong dau ra.
model.eval()

for idx in range(min(3, len(test_rows))):
    prompt_messages = test_rows[idx]['messages'][:-1]
    reference = test_rows[idx]['messages'][-1]['content']

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('=' * 100)
    print('HOLDOUT EXAMPLE', idx)
    print('- Prompt preview:')
    print(prompt_messages[-1]['content'][:500])
    print('- Prediction preview:')
    print(generated[:1200])
    print('- Reference preview:')
    print(reference[:1200])


In [ ]:
# Zip adapter de tai ve nhanh hon.
import shutil

archive_base = OUT_DIR.parent / f'{OUT_DIR.name}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(OUT_DIR))
print('Created:', archive_path)


## Ghi chu van hanh

- Neu `bitsandbytes` loi tren Windows native, chay notebook nay trong WSL2 Ubuntu tren cung may RTX 5090.
- `test.jsonl` phai giu nguyen lam holdout, khong dung de tune hyperparameter.
- Sau khi lay adapter ve may local, ban co the merge/evaluate rieng theo pipeline cua repo.
- Neu train loss giam manh nhung output holdout te hon, giam `num_train_epochs` xuong 2 hoac giam `learning_rate` xuong `5e-5`.